# Jigsaw Puzzle Solving with JigNet & JigSolver
This notebook downloads a subset from COCO benchmark and resizes the images (512, 512, 3)



## 1. Environment Setup & Data Acquisition

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Installing Dependencies
We install `pycocotools` for managing COCO dataset downloads and annotations.

In [ ]:
pip install pycocotools requests

### Downloading COCO Images and Annotations
We fetch 250 random validation images from the MS COCO 2017 dataset to form our benchmark jigsaw pool.

In [ ]:
import os
import random
import urllib.request
from pycocotools.coco import COCO

# 1. Download the small COCO validation annotations file (~19MB)
ann_url = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
ann_zip = "annotations_trainval2017.zip"

if not os.path.exists("annotations"):
    print("Downloading annotations index...")
    urllib.request.urlretrieve(ann_url, ann_zip)
    import zipfile

    with zipfile.ZipFile(ann_zip, "r") as zip_ref:
        zip_ref.extractall(".")
    os.remove(ann_zip)

# 2. Initialize COCO API & select 250 random image IDs
coco = COCO("annotations/instances_val2017.json")
img_ids = coco.getImgIds()
random_img_ids = random.sample(img_ids, 250)
images = coco.loadImgs(random_img_ids)

# 3. Create output directory and download images
output_dir = "/content/drive/MyDrive/Jigsaw_Puzzle_Project/jigsaw_dataset/COCO"
os.makedirs(output_dir, exist_ok=True)

print("Downloading 250 images...")
for img_info in images:
    img_url = img_info["coco_url"]
    file_name = img_info["file_name"]
    save_path = os.path.join(output_dir, file_name)

    urllib.request.urlretrieve(img_url, save_path)

print(f"Done! 250 images saved to '{output_dir}'.")

loading annotations into memory...
Done (t=1.01s)
creating index...
index created!
Done! 500 images saved to '/content/drive/MyDrive/Jigsaw_Puzzle_Project/jigsaw_dataset/COCO'.


## 2. Image Preprocessing & Tile Splitting
To prepare the images for a 4x4 grid jigsaw task (each tile size $128 \times 128$), we first standardize the downloaded COCO images to a fixed resolution of $512 \times 512$ pixels.

In [ ]:
import cv2
import numpy as np

def resize_image(image):
    """
    Shrink or stretch an RGB image to ensure its size is (512, 512, 3).

    Parameters:
    image (numpy.ndarray): The input image of shape (H, W, 3)

    Returns:
    numpy.ndarray: The resized image of shape (512, 512, 3)
    """
    # Ensure the image has 3 channels
    if len(image.shape) == 2:
        # Convert grayscale to RGB if necessary
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    elif image.shape[2] == 4:
        # Convert RGBA to RGB if necessary
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)

    # Resize to (512, 512)
    resized_image = cv2.resize(image, (512, 512), interpolation=cv2.INTER_LINEAR)
    return resized_image

### Standardizing Directories
We read original files and save the standardized versions to Google Drive.

In [ ]:
import os
import cv2

def process_and_resize_folder(in_dir, out_dir):
    """
    Reads all images from in_dir, resizes them to (512, 512, 3) using resize_image,
    and saves them to out_dir.
    """
    if not os.path.exists(out_dir):
        os.makedirs(out_dir, exist_ok=True)
        print(f"Created output directory: {out_dir}")

    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff')
    files = [f for f in os.listdir(in_dir) if f.lower().endswith(valid_extensions)]

    print(f"Found {len(files)} images to process from '{in_dir}'.")

    for idx, filename in enumerate(files, 1):
        img_path = os.path.join(in_dir, filename)
        # Read image using OpenCV
        img = cv2.imread(img_path)
        if img is None:
            print(f"Skipping {filename}, could not read image.")
            continue

        # Convert BGR (OpenCV default) to RGB for resize_image
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Resize image
        resized_rgb = resize_image(img_rgb)

        # Convert RGB back to BGR for saving with OpenCV
        resized_bgr = cv2.cvtColor(resized_rgb, cv2.COLOR_RGB2BGR)

        # Save resized image to out_dir
        save_path = os.path.join(out_dir, filename)
        cv2.imwrite(save_path, resized_bgr)

        if idx % 50 == 0 or idx == len(files):
            print(f"Processed and saved {idx}/{len(files)} images.")

In [ ]:
in_dir = '/content/drive/MyDrive/Jigsaw_Puzzle_Project/jigsaw_dataset/COCO/original_images'
out_dir = '/content/drive/MyDrive/Jigsaw_Puzzle_Project/jigsaw_dataset/COCO/resized_images'
process_and_resize_folder(in_dir, out_dir)

Found 250 images to process from '/content/drive/MyDrive/Jigsaw_Puzzle_Project/jigsaw_dataset/COCO/original_images'.
Processed and saved 50/250 images.
Processed and saved 100/250 images.
Processed and saved 150/250 images.
Processed and saved 200/250 images.
Processed and saved 250/250 images.
